# Notebook 06 — Modelling

Fits and saves the three persisted models (`src/modeling/train.py`, the same code
`python -m src.modeling.train` runs): Linear Regression and Ridge (the `linear` feature
set) and Gradient Boosting (the `gradient_boosting` feature set, full zone identity).

LASSO Regression is **not** fit here — it's refit inline in notebook `07` alongside its
alpha tuning, since it was never persisted as a `.joblib` in the original pipeline either.
No evaluation happens in this notebook (Table 3/4/5's metrics are notebook `07`'s job) —
this is purely fit + save, matching the existing scope split between Phase 10 (fitting)
and Phase 11 (evaluation).

## 1. Setup — load Processed data, chronological split

In [1]:
from src.features import build_feature_matrix, split_train_test
from src.geospatial import load_processed_zone_hour
from src.modeling.train import (
    fit_gradient_boosting,
    fit_linear_regression,
    fit_ridge_regression,
    save_model,
)
from src.config import MODELS_DIR

processed_df = load_processed_zone_hour()
train_df, test_df = split_train_test(processed_df)
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")

2026-08-28 16:15:57.090 | INFO     | src.config:<module>:26 - PROJ_ROOT path is: /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final


Train rows: 9,228,144 | Test rows: 1,906,224


## 2. Build feature matrices

One per feature-set family: `linear` (borough + baseline, shared by Linear Regression and
Ridge) and `gradient_boosting` (full zone identity).

In [2]:
X_train_linear, y_train_linear = build_feature_matrix(train_df, "linear")
X_train_gb, y_train_gb = build_feature_matrix(train_df, "gradient_boosting")

print(f"[linear]            X_train {X_train_linear.shape}")
print(f"[gradient_boosting] X_train {X_train_gb.shape}")

[linear]            X_train (9215520, 65)
[gradient_boosting] X_train (9215520, 17)


## 3. Fit + save Linear Regression

In [3]:
linear_model = fit_linear_regression(X_train_linear, y_train_linear)
save_model(linear_model, MODELS_DIR / "linear_regression.joblib")
print("Linear Regression fit and saved.")

2026-08-28 06:16:16.367 | INFO     | src.modeling.train:save_model:180 - Saved model to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/models/linear_regression.joblib
Linear Regression fit and saved.


## 4. Fit + save Ridge Regression

In [4]:
ridge_model = fit_ridge_regression(X_train_linear, y_train_linear)
save_model(ridge_model, MODELS_DIR / "ridge.joblib")
print("Ridge Regression fit and saved.")

2026-08-28 06:16:18.757 | INFO     | src.modeling.train:save_model:180 - Saved model to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/models/ridge.joblib
Ridge Regression fit and saved.


## 5. Fit + save Gradient Boosting

Must use the `gradient_boosting` feature set's native pandas-categorical encoding, not
one-hot — one-hot-encoding `pu_location_id` (263 columns) made 30 boosting iterations
alone take over 2 hours in a real timing test.

In [5]:
gb_model = fit_gradient_boosting(X_train_gb, y_train_gb)
save_model(gb_model, MODELS_DIR / "gradient_boosting.joblib")
print("Gradient Boosting fit and saved.")

2026-08-28 06:16:35.369 | INFO     | src.modeling.train:save_model:180 - Saved model to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/models/gradient_boosting.joblib
Gradient Boosting fit and saved.


## Notes

- Output: `models/linear_regression.joblib`, `models/ridge.joblib`,
  `models/gradient_boosting.joblib`.
- Hyperparameters are first-pass/undecided values (sklearn defaults + a fixed
  `random_state`), not tuned — see `src/modeling/train.py` module docstring.
- LASSO Regression, and every model's accuracy/ranking evaluation (Tables 3–5, the
  reliability table, DTDI), is notebook `07`.